<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/main/Daily_challenge_W5_J2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Défi quotidien : Classification avec des réseaux neuronaux dans TensorFlow

## 1. Comprendre les types de classification

La classification est une tâche d'apprentissage supervisé qui consiste à attribuer une étiquette de classe à un nouvel échantillon d'entrée, sur la base des modèles appris à partir des données d'entraînement. Il existe plusieurs types de classification :

### Classification Binaire

La classification binaire est la tâche la plus simple, où les données sont classées en deux classes distinctes. L'objectif est de prédire l'une des deux catégories possibles.

**Exemple :** Prédire si un e-mail est du spam (1) ou non (0).

### Classification Multiclasse

La classification multiclasse est une tâche où les données sont classées en trois classes ou plus, mais chaque échantillon appartient à *une seule* de ces classes.

**Exemple :** Classer une image d'animal comme 'chien', 'chat' ou 'oiseau'. Une image ne peut pas être un chien et un chat à la fois.

### Classification Multi-étiquettes

La classification multi-étiquettes est une tâche où chaque échantillon peut être associé à *plusieurs* étiquettes de classe simultanément. Contrairement à la classification multiclasse, les classes ne sont pas mutuellement exclusives.

**Exemple :** Classer un film par genre. Un film peut être à la fois 'Action', 'Science-fiction' et 'Aventure'.

## 2. Configurez votre environnement Python et votre jeu de données.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

print(f"TensorFlow version: {tf.__version__}")

### Création de l'ensemble de données `make_circles`

In [ ]:
samples = 1000
X, y = make_circles(samples,
                    noise = 0.03,
                    random_state = 42)

print('X (premières 5 lignes) : ')
print(X[:5])
print('\ny (premières 5 lignes) : ')
print(y[:5])

### Visualisation de l'ensemble de données

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu)
plt.title("Visualisation de l'ensemble de données make_circles")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

## 3. Construire un modèle de réseau neuronal de base

In [ ]:
# Définir le modèle de base
model_0 = tf.keras.Sequential([
    tf.keras.layers.Dense(1, activation='sigmoid') # Une seule couche dense avec une activation sigmoïde pour la classification binaire
])

# Compiler le modèle
model_0.compile(loss=tf.keras.losses.BinaryCrossentropy(),
              optimizer=tf.keras.optimizers.SGD(), # Optimiseur par défaut
              metrics=['accuracy'])

# Entraîner le modèle
history_0 = model_0.fit(X, y, epochs=100, verbose=0) # verbose=0 pour ne pas afficher la progression de l'entraînement

# Vérifier la précision
loss_0, accuracy_0 = model_0.evaluate(X, y)
print(f"Précision du modèle de base sur l'ensemble complet: {accuracy_0*100:.2f}%")
print(f"Perte du modèle de base sur l'ensemble complet: {loss_0:.4f}")

## 4. Améliorer le modèle

In [ ]:
# Définir un modèle amélioré avec plus de couches et de neurones
model_1 = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='relu'), # Première couche cachée avec 10 neurones et ReLU
    tf.keras.layers.Dense(5, activation='relu'),  # Deuxième couche cachée avec 5 neurones et ReLU
    tf.keras.layers.Dense(1, activation='sigmoid')  # Couche de sortie pour la classification binaire
])

# Compiler le modèle avec l'optimiseur Adam
model_1.compile(loss=tf.keras.losses.BinaryCrossentropy(),
              optimizer=tf.keras.optimizers.Adam(), # Utilisation de l'optimiseur Adam
              metrics=['accuracy'])

# Entraîner le modèle pour plus d'époques
history_1 = model_1.fit(X, y, epochs=200, verbose=0) # Entraînement sur 200 époques

# Évaluer le modèle
loss_1, accuracy_1 = model_1.evaluate(X, y)
print(f"Précision du modèle amélioré sur l'ensemble complet: {accuracy_1*100:.2f}%")
print(f"Perte du modèle amélioré sur l'ensemble complet: {loss_1:.4f}")

## 5. Visualiser la frontière de décision

In [ ]:
def plot_decision_boundary(model, X, y):
    """
    Trace la frontière de décision générée par un modèle de classification.
    """
    # Définir la plage de l'axe X et Y
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))

    # Créer un ensemble de données pour faire des prédictions sur toute la grille
    X_grid = np.c_[xx.ravel(), yy.ravel()]

    # Faire des prédictions (probabilités)
    y_pred_proba = model.predict(X_grid)

    # Convertir les probabilités en classes binaires (0 ou 1)
    # Cela dépend de la fonction d'activation de la dernière couche. Pour sigmoid, 0.5 est un seuil courant.
    if y_pred_proba.ndim > 1 and y_pred_proba.shape[1] > 1: # Pour la classification multiclasse avec softmax
        y_pred = np.argmax(y_pred_proba, axis=1)
    else: # Pour la classification binaire avec sigmoid
        y_pred = np.round(y_pred_proba).flatten()

    # Remodeler les prédictions en grille
    Z = y_pred.reshape(xx.shape)

    # Tracer la frontière de décision et les points de données
    plt.contourf(xx, yy, Z, cmap=plt.cm.RdYlBu, alpha=0.7)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu, s=40, edgecolors='k')
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title("Frontière de décision du modèle")
    plt.show()


# Visualiser la frontière de décision du modèle de base
print("Frontière de décision pour le Modèle de Base:")
plt.figure(figsize=(8, 6))
plot_decision_boundary(model_0, X, y)

# Visualiser la frontière de décision du modèle amélioré
print("Frontière de décision pour le Modèle Amélioré:")
plt.figure(figsize=(8, 6))
plot_decision_boundary(model_1, X, y)

## 6. Incorporer des fonctions d'activation

Les fonctions d'activation sont essentielles dans les réseaux neuronaux car elles introduisent des non-linéarités, permettant au modèle d'apprendre des relations complexes dans les données. Voici deux fonctions d'activation courantes :

*   **ReLU (Rectified Linear Unit)** : `tf.keras.activations.relu` ou simplement `'relu'`
    *   `f(x) = max(0, x)`
    *   Avantages : Simple, rapide à calculer, aide à atténuer le problème de la disparition du gradient.
    *   Inconvénients : Peut souffrir du problème des 'neurones morts' (si l'entrée est toujours négative, le neurone ne s'active jamais).

*   **Sigmoïde** : `tf.keras.activations.sigmoid` ou `'sigmoid'`
    *   `f(x) = 1 / (1 + exp(-x))`
    *   Avantages : Comprime les valeurs d'entrée entre 0 et 1, ce qui la rend utile pour la couche de sortie dans la classification binaire (interprétée comme une probabilité).
    *   Inconvénients : Souffre du problème de la disparition du gradient pour les grandes valeurs positives ou négatives, et sa sortie n'est pas centrée autour de zéro, ce qui peut rendre l'entraînement plus difficile.

In [ ]:
# Le modèle_1 utilise déjà ReLU pour les couches cachées et Sigmoid pour la couche de sortie.
# C'est une bonne configuration pour ce type de problème.

# Nous allons recréer un modèle similaire pour le comparer spécifiquement, si besoin.
model_2 = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(10, activation='relu'), # Ajout d'une couche supplémentaire avec ReLU
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model_2.compile(loss=tf.keras.losses.BinaryCrossentropy(),
                optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                metrics=['accuracy'])

history_2 = model_2.fit(X, y, epochs=250, verbose=0)

loss_2, accuracy_2 = model_2.evaluate(X, y)
print(f"Précision du modèle avec fonctions d'activation spécifiques sur l'ensemble complet: {accuracy_2*100:.2f}%")
print(f"Perte du modèle avec fonctions d'activation spécifiques sur l'ensemble complet: {loss_2:.4f}")

print("Frontière de décision pour le Modèle avec Fonctions d'Activation Spécifiques:")
plt.figure(figsize=(8, 6))
plot_decision_boundary(model_2, X, y)

## 7. Diviser les données en ensembles d'entraînement et de test

In [ ]:
# Diviser les données en ensembles d'entraînement et de test (80% entraînement, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Taille de l'ensemble d'entraînement X: {X_train.shape}")
print(f"Taille de l'ensemble de test X: {X_test.shape}")
print(f"Taille de l'ensemble d'entraînement y: {y_train.shape}")
print(f"Taille de l'ensemble de test y: {y_test.shape}")

# Entraîner le modèle amélioré (model_2) sur l'ensemble d'entraînement
history_final = model_2.fit(X_train, y_train, epochs=300, verbose=0)

# Évaluer les performances du modèle sur l'ensemble de test
loss_final, accuracy_final = model_2.evaluate(X_test, y_test)
print(f"Précision finale du modèle sur l'ensemble de test: {accuracy_final*100:.2f}%")
print(f"Perte finale du modèle sur l'ensemble de test: {loss_final:.4f}")

## 8. Évaluer et visualiser les performances du modèle final

In [ ]:
# Évaluer le modèle sur l'ensemble d'entraînement
loss_train, accuracy_train = model_2.evaluate(X_train, y_train, verbose=0)
print(f"Précision du modèle sur l'ensemble d'entraînement: {accuracy_train*100:.2f}%")
print(f"Perte du modèle sur l'ensemble d'entraînement: {loss_train:.4f}")

# Évaluer le modèle sur l'ensemble de test
loss_test, accuracy_test = model_2.evaluate(X_test, y_test, verbose=0)
print(f"Précision du modèle sur l'ensemble de test: {accuracy_test*100:.2f}%")
print(f"Perte du modèle sur l'ensemble de test: {loss_test:.4f}")

# Visualiser les prédictions pour les données d'entraînement
print("Frontière de décision pour les données d'entraînement:")
plt.figure(figsize=(8, 6))
plot_decision_boundary(model_2, X_train, y_train)

# Visualiser les prédictions pour les données de test
print("Frontière de décision pour les données de test:")
plt.figure(figsize=(8, 6))
plot_decision_boundary(model_2, X_test, y_test)

## 9. Résumer les points clés

Au cours de cet exercice, nous avons exploré les principes fondamentaux de la classification en utilisant les réseaux neuronaux avec TensorFlow. Voici les points clés que nous avons appris :

*   **Types de Classification** : Nous avons compris la distinction entre la classification binaire, multiclasse et multi-étiquettes, chacune ayant ses propres cas d'utilisation.
*   **Construction de Modèles Séquentiels** : Nous avons construit des réseaux neuronaux avec des couches denses, en commençant par un modèle simple et en l'améliorant progressivement.
*   **Fonctions d'Activation** : L'importance des fonctions d'activation comme ReLU pour les couches cachées et Sigmoïde pour la couche de sortie dans la classification binaire a été démontrée. Elles introduisent la non-linéarité, permettant au modèle d'apprendre des motifs complexes.
*   **Optimiseurs et Fonctions de Perte** : Nous avons utilisé l'entropie croisée binaire comme fonction de perte standard pour la classification binaire et avons expérimenté des optimiseurs comme SGD et Adam, Adam montrant généralement de meilleures performances.
*   **Visualisation des Données et des Frontières de Décision** : La visualisation de l'ensemble de données initial et des frontières de décision des modèles est cruciale pour comprendre comment le modèle sépare les classes et pour diagnostiquer les problèmes de performance (sous-apprentissage, sur-apprentissage).
*   **Importance du Réglage des Hyperparamètres** : L'ajout de plus de couches, l'augmentation du nombre de neurones, le choix des fonctions d'activation et le réglage du nombre d'époques ont tous contribué à améliorer la capacité du modèle à résoudre un problème non linéaire comme l'ensemble de données `make_circles`.
*   **Division Entraînement/Test** : Il est essentiel de diviser les données en ensembles d'entraînement et de test pour évaluer objectivement les performances du modèle sur des données non vues et éviter le sur-apprentissage.

En résumé, la construction de modèles de classification efficaces implique une compréhension des différents types de problèmes, une conception appropriée de l'architecture du réseau neuronal, le choix judicieux des fonctions d'activation et des optimiseurs, et une évaluation rigoureuse à l'aide de techniques de visualisation et de validation croisée.